# Load dataset

In [ ]:
# Use the visual_retriever project (run from repo root: cd visual_retriever && uv sync, then select that .venv as kernel)
import sys
from pathlib import Path
_root = Path(__file__).resolve().parents[1] if "__file__" in dir() else Path.cwd().parent
if str(_root / "visual_retriever") not in sys.path:
    sys.path.insert(0, str(_root / "visual_retriever"))

In [1]:
import torch
from PIL import Image

# --- CONFIGURATION ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [4]:
from visual_retriever.dataset import load_data_vidore

subset = "physics"
lang = "english"
ds_corpus, ds_queries, ds_qrels, ds_metadata = load_data_vidore(subset, lang)



In [5]:
num_queries = len(ds_queries)
num_corpus = len(ds_corpus)
num_qrels = len(ds_qrels)
num_metadata = len(ds_metadata)

print(f"Number of queries: {num_queries}")
print(f"Number of corpus: {num_corpus}")


Number of queries: 302
Number of corpus: 1674


# Encode images

## Load model

In [6]:
from visual_retriever.model import load_visual_retriever_model
model = load_visual_retriever_model()


Loading weights: 100%|██████████| 916/916 [00:00<00:00, 1149.30it/s, Materializing param=vision_model.vision_model.post_layernorm.weight]                      


## Pre-compute and save image embeddings

In [7]:
from visual_retriever.features import precompute_image_embeddings
save_dir = "colembed_cache_pages_fp16"
precompute_image_embeddings(model, ds_corpus, save_dir=save_dir)

100%|██████████| 53/53 [00:40<00:00,  1.32it/s]


# Pre-compute and save query embeddings

In [10]:
from visual_retriever.features import precompute_query_embeddings
save_dir = "colembed_cache_queries_fp16"
precompute_query_embeddings(model, ds_queries, save_dir=save_dir)


Pre-computing query embeddings: 0it [00:00, ?it/s]/home/fidaa/.cache/huggingface/modules/transformers_modules/nvidia/llama_hyphen_nemotron_hyphen_colembed_hyphen_vl_hyphen_3b_hyphen_v2/680b47b199f99bc0ec2f4e90ffa583ec0c2e452c/modeling_llama_nemotron_vl.py:173: FutureWarning: `input_embeds` is deprecated and will be removed in version 5.6.0 for `create_bidirectional_mask`. Use `inputs_embeds` instead.
  return create_bidirectional_mask(
Extracting query embeddings...: 100%|██████████| 1/1 [00:00<00:00, 61.59it/s]
Pre-computing query embeddings: 302it [00:08, 35.57it/s]


# Compute ndcg@10

In [8]:
from visual_retriever.features import load_precomputed_image_embeddings
save_dir = "colembed_cache_pages_fp16"
pages_embeddings = load_precomputed_image_embeddings(ds_corpus, save_dir)


Loading pages embeddings: 100%|██████████| 1674/1674 [00:27<00:00, 61.38it/s] 


In [11]:
from visual_retriever.features import load_precomputed_query_embeddings
save_dir = "colembed_cache_queries_fp16"
query_embeddings = load_precomputed_query_embeddings(ds_queries, save_dir)


Loading query embeddings: 100%|██████████| 302/302 [00:00<00:00, 2380.91it/s]


In [13]:
from visual_retriever.utils import evaluate_ndcg

idx_to_corpus_id = [ds_corpus[i]["corpus_id"] for i in range(len(ds_corpus))]
qrels = [
    {"query_id": q, "corpus_id": c, "score": s}
    for q, c, s in zip(ds_qrels["query_id"], ds_qrels["corpus_id"], ds_qrels["score"])
]


ndcg_at_k = evaluate_ndcg(model, query_embeddings, ds_queries, ds_qrels, pages_embeddings, idx_to_corpus_id, k=10)

TypeError: unhashable type: 'dict'